# 03 — Sequence Models: MLP, LSTM & Hybrid (PyTorch)
Train and compare three neural network architectures on temporal
windows of SOLEY PV data:

| Architecture | Description |
|---|---|
| **MLP**    | Per-timestep features from the last step of the window |
| **LSTM**   | Bi-directional LSTM over the full window |
| **Hybrid** | MLP branch (last step) + LSTM branch fused (recommended) |

The streaming memmap pipeline keeps peak RAM ≈ 1 parquet file (~60–100 MB)
regardless of dataset size.


## 1. Imports & setup

In [ ]:
import sys, pathlib, shutil
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from library.utils      import setup_logging, get_device, get_num_workers
from library.config     import BatchConfig
from library.data       import prepare_file_registry, assign_splits
from library.models.trainer import run_pytorch_task

setup_logging()
%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## 2. Configuration

In [ ]:
DATA_DIR   = "data"
OUTPUT_DIR = "outputs/sequence_models"
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Training hyperparameters
WINDOW_SIZE = 288    # timesteps per sample (288 × 5 min = 24 h)
STRIDE      = 12     # step between windows (12 × 5 min = 1 h)
BATCH_SIZE  = 1024
EPOCHS      = 30
PATIENCE    = 7
MAX_RUNS    = 96     # max parquet files (set to None for all)

# Model variants to train: "mlp", "lstm", "hybrid", or all three
MODEL_MODES = ["mlp", "lstm", "hybrid"]


In [ ]:
cfg    = BatchConfig(DATA_DIR)
device = get_device()
n_workers = get_num_workers()
print(f"Device:  {device}")
print(f"Workers: {n_workers}")
print(cfg)


## 3. Build file registry & assign splits

In [ ]:
# No data is loaded here — just file paths and fault-type metadata
registry = prepare_file_registry(DATA_DIR, cfg, max_runs=MAX_RUNS)
assign_splits(registry)

fault_types = sorted({e['fault_type'] for e in registry})
print(f"\nFault types in registry: {fault_types}")


## 4. Feature list

In [ ]:
eng = ["hour_sin", "hour_cos", "doy_sin", "doy_cos",
       "performance_ratio", "pr_deviation", "dc_ac_power_ratio", "power_step"]
feature_list = list(cfg.scada_features) + eng + list(cfg.device_features) + list(cfg.stress_features)
print(f"Total features: {len(feature_list)}")


## 5. Task A — Fault Detection (binary)

In [ ]:
detection_reports = run_pytorch_task(
    task_name   = "Fault Detection",
    target_col  = "fault_active",
    registry    = registry,
    feature_cols= feature_list,
    model_modes = MODEL_MODES,
    window_size = WINDOW_SIZE,
    stride      = STRIDE,
    batch_size  = BATCH_SIZE,
    epochs      = EPOCHS,
    device      = device,
    output_dir  = OUTPUT_DIR,
    patience    = PATIENCE,
    cfg         = cfg,
    num_workers = n_workers,
)


In [ ]:
# Training curves
from IPython.display import Image, display as ipy_display
for mode in MODEL_MODES:
    p = f"{OUTPUT_DIR}/curves_fault_detection_{mode}.png"
    if pathlib.Path(p).exists():
        print(f"\n{mode.upper()} training curves:")
        ipy_display(Image(p))


## 6. Task B — Fault Classification (multi-class)

In [ ]:
classification_reports = run_pytorch_task(
    task_name   = "Fault Classification",
    target_col  = "fault_type",
    registry    = registry,
    feature_cols= feature_list,
    model_modes = MODEL_MODES,
    window_size = WINDOW_SIZE,
    stride      = STRIDE,
    batch_size  = BATCH_SIZE,
    epochs      = EPOCHS,
    device      = device,
    output_dir  = OUTPUT_DIR,
    patience    = PATIENCE,
    cfg         = cfg,
    num_workers = n_workers,
)


In [ ]:
# Per-fault F1 comparison
p = f"{OUTPUT_DIR}/per_fault_f1_fault_classification.png"
if pathlib.Path(p).exists():
    ipy_display(Image(p))


## 7. Confusion matrices

In [ ]:
for task_tag in ["fault_detection", "fault_classification"]:
    for mode in MODEL_MODES:
        p = f"{OUTPUT_DIR}/confusion_{task_tag}_{mode}.png"
        if pathlib.Path(p).exists():
            print(f"\n{task_tag} — {mode.upper()}")
            ipy_display(Image(p))


## 8. Saved artifacts

In [ ]:
import os
outputs = sorted(pathlib.Path(OUTPUT_DIR).iterdir())
for f in outputs:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:<50s}  {size_mb:.2f} MB")
